# Phase 1: Data Pipeline and Baseline

**Research question (project-level):** can a transformer trained on raw EEG time-series match established CNN baselines on sleep stage classification, and where does it fail?

This notebook builds and validates the foundation everything else depends on: a correct, leakage-free preprocessing pipeline, and a small CNN baseline to beat. It does not try to be clever — the point of a baseline is to be defensible, not optimal.

**Dataset:** Sleep-EDF Expanded (PhysioNet), single channel (Fpz-Cz), 30-second epochs, 5-class labels (W, N1, N2, N3, REM).

**What this notebook does:**
1. Sanity-checks the preprocessing on a couple of subjects (shapes, one example epoch per stage).
2. Builds the full subject-level train/val/test split.
3. Trains the baseline CNN.
4. Reports macro F1, per-class F1, and a confusion matrix on the held-out test set.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, f1_score
from torch.utils.data import DataLoader

from src.baseline import CNNBaseline
from src.config import (
    BATCH_SIZE,
    CASSETTE_DIR,
    LABEL_NAMES,
    SFREQ,
    get_device,
    set_seed,
)
from src.data import (
    SleepEDFDataset,
    build_epoch_arrays,
    discover_subjects,
    get_subject_split,
    load_subject_epochs,
)
from src.train import evaluate, train_model

set_seed()
device = get_device()
print(f"device: {device}")

## 1. Sanity check: preprocessing on a couple of subjects

Before trusting the pipeline on the full dataset, confirm on a small sample that: files load, the hypnogram aligns with the signal, epochs come out at the expected length, and the label distribution looks like real sleep architecture (mostly W/N2, N1 rare).

In [ ]:
subjects = discover_subjects(CASSETTE_DIR)
print(f"found {len(subjects)} subjects with both PSG and Hypnogram files")
for s in subjects[:5]:
    print(f"  subject {s.subject_id}: {s.psg_path.name}  /  {s.hyp_path.name}")

assert len(subjects) >= 1, (
    "no subjects found under data/sleep-cassette/ — run "
    "scripts/download_data.sh --sample first (or the full download)"
)

In [ ]:
sample_epochs, sample_labels = load_subject_epochs(subjects[0])
print(f"epochs shape: {sample_epochs.shape}  (n_epochs, samples_per_epoch)")
print(f"labels shape: {sample_labels.shape}")

unique, counts = np.unique(sample_labels, return_counts=True)
print("\nlabel distribution for this subject:")
for label_id, count in zip(unique, counts):
    print(f"  {LABEL_NAMES[label_id]:>4}: {count:4d} epochs ({100 * count / len(sample_labels):.1f}%)")

**Figure: one example 30-second epoch per sleep stage, from the sample subject.**

What to notice: Wake (W) and REM both show low-amplitude, mixed-frequency activity and can look deceptively similar at a glance — this is the classic confusion pair we'll revisit in Phase 3. N2 typically shows visible sleep spindles / K-complexes (sharp transient bumps). N3 (slow-wave sleep) has large, slow, high-amplitude oscillations — the easiest stage to spot by eye. If any of these traces look like flat lines or pure noise, the filtering/resampling step likely has a bug.

In [ ]:
fig, axes = plt.subplots(len(LABEL_NAMES), 1, figsize=(10, 8), sharex=True)
time_axis = np.arange(sample_epochs.shape[1]) / SFREQ

for label_id, label_name in enumerate(LABEL_NAMES):
    idx = np.where(sample_labels == label_id)[0]
    ax = axes[label_id]
    if len(idx) == 0:
        ax.set_title(f"{label_name}: not present in this subject")
        ax.axis("off")
        continue
    ax.plot(time_axis, sample_epochs[idx[0]], linewidth=0.6)
    ax.set_title(label_name)
    ax.set_ylabel("\u00b5V")

axes[-1].set_xlabel("time (s)")
fig.suptitle("One example epoch per sleep stage")
fig.tight_layout()
plt.show()

## 2. Subject-level train/val/test split

Split is 60/20/20 by **subject**, not by epoch — see `get_subject_split` in `src/data.py` for why an epoch-level split would leak. All epochs are loaded and concatenated per split into flat arrays, then wrapped in `SleepEDFDataset` / `DataLoader`.

In [ ]:
subject_ids = [s.subject_id for s in subjects]
train_ids, val_ids, test_ids = get_subject_split(subject_ids)
print(f"train subjects ({len(train_ids)}): {train_ids}")
print(f"val subjects   ({len(val_ids)}): {val_ids}")
print(f"test subjects  ({len(test_ids)}): {test_ids}")

train_epochs, train_labels, _ = build_epoch_arrays(subjects, train_ids)
val_epochs, val_labels, _ = build_epoch_arrays(subjects, val_ids)
test_epochs, test_labels, test_subject_ids = build_epoch_arrays(subjects, test_ids)

print(f"\ntrain epochs: {train_epochs.shape[0]}")
print(f"val epochs:   {val_epochs.shape[0]}")
print(f"test epochs:  {test_epochs.shape[0]}")

train_loader = DataLoader(SleepEDFDataset(train_epochs, train_labels), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SleepEDFDataset(val_epochs, val_labels), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(SleepEDFDataset(test_epochs, test_labels), batch_size=BATCH_SIZE, shuffle=False)

## 3. Train the baseline CNN

Small model, deliberately: two conv1d blocks, global average pool, linear head (see `src/baseline.py`). Adam + cross-entropy, early stopping on validation loss (see `src/train.py`). This is the reference the transformer in Phase 2 has to beat under identical training conditions.

In [ ]:
model = CNNBaseline()
model, history = train_model(model, train_loader, val_loader, device=device)

**Figure: training curves.** Left: train vs. validation loss per epoch — divergence between them would indicate overfitting. Right: validation accuracy per epoch. Early stopping restores the checkpoint with the lowest validation loss, not necessarily the last epoch shown.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.train_loss, label="train")
ax1.plot(history.val_loss, label="val")
ax1.set_xlabel("epoch")
ax1.set_ylabel("loss")
ax1.set_title("Loss")
ax1.legend()

ax2.plot(history.val_acc, color="green")
ax2.set_xlabel("epoch")
ax2.set_ylabel("accuracy")
ax2.set_title("Validation accuracy")

fig.tight_layout()
plt.show()

## 4. Evaluate on the held-out test set

Macro F1 is the headline metric (averages per-class F1 equally, so the rare N1 class isn't washed out by W/N2 dominance). Per-class F1 and the confusion matrix show where the errors actually are — a preview of the full Phase 3 error analysis.

In [ ]:
results = evaluate(model, test_loader, device=device)
preds, labels = results["preds"], results["labels"]

macro_f1 = f1_score(labels, preds, average="macro")
per_class_f1 = f1_score(labels, preds, average=None, labels=list(range(len(LABEL_NAMES))))

print(f"macro F1: {macro_f1:.4f}\n")
print("per-class F1:")
for name, f1 in zip(LABEL_NAMES, per_class_f1):
    print(f"  {name:>4}: {f1:.4f}")

**Figure: normalized confusion matrix on the test set.** Rows are true labels, columns are predicted labels, each row sums to 1.0. Off-diagonal mass shows which stages get mistaken for which — watch particularly for N1 bleeding into W or N2, a well-known failure mode in sleep staging that we quantify properly in Phase 3.

In [ ]:
cm = confusion_matrix(labels, preds, normalize="true")
disp = ConfusionMatrixDisplay(cm, display_labels=LABEL_NAMES)
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
ax.set_title("Baseline CNN: normalized confusion matrix (test set)")
plt.show()

## Next steps

Phase 2 (`notebooks/02_transformer.ipynb`) trains a patch-based transformer under identical conditions (same `train.py` loop, same optimizer/schedule) and compares macro F1 head-to-head against this baseline. Phase 3 does the structured error analysis that's the actual point of the project.